**Install Required Libraries**

In [ ]:
!pip -q install -U transformers datasets accelerate peft trl sentencepiece evaluate rouge_score bert-score huggingface_hub

**Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Import Core Libraries and Set Seed**

In [ ]:
import os
import json
import math
import random
import numpy as np
import pandas as pd
import torch

from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)


**Define Model, Dataset, and Training Paths**

In [ ]:
# Define the base Gemma instruction-tuned model and the Hugging Face dataset repository
MODEL_ID = "google/gemma-1.1-2b-it"
DATASET_REPO = "BSVGK/drugbank_dataset"

# Specify the train, validation, and test JSONL files used for LoRA fine-tuning
DATA_FILES = {
    "train": "kg_to_text_train.jsonl",
    "validation": "kg_to_text_validation.jsonl",
    "test": "kg_to_text_test.jsonl"
}

# Define output paths for saving LoRA checkpoints, adapter weights, and merged final model
OUTPUT_DIR = "/content/drive/MyDrive/Depixen/gemma-1.1-2b-it-drugbank-kg2text/LORA_OUTPUTS"
ADAPTER_DIR = f"{OUTPUT_DIR}/adapter"
MERGED_DIR = f"{OUTPUT_DIR}/merged"

# Create all required output directories if they do not already exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

# Set the shared training hyperparameters for LoRA fine-tuning
COMMON_EPOCHS = 3
COMMON_LR = 2e-4
COMMON_WEIGHT_DECAY = 0.01
COMMON_WARMUP_RATIO = 0.03
COMMON_TRAIN_BATCH_SIZE = 2
COMMON_EVAL_BATCH_SIZE = 1
COMMON_GRAD_ACCUM = 8
COMMON_LOGGING_STEPS = 10
COMMON_EVAL_STEPS = 50
COMMON_SAVE_STEPS = 50
COMMON_SAVE_TOTAL_LIMIT = 2

# Set the shared LoRA adapter configuration values
COMMON_LORA_R = 16
COMMON_LORA_ALPHA = 32
COMMON_LORA_DROPOUT = 0.05

**Load Dataset from Hugging Face**

In [ ]:
from datasets import load_dataset

raw_ds = load_dataset(
    DATASET_REPO,
    data_files=DATA_FILES,
    token=HF_TOKEN
)

print(raw_ds)

**Preview Dataset Sizes and Sample Records**


In [ ]:
print("Train size:", len(raw_ds["train"]))
print("Validation size:", len(raw_ds["validation"]))
print("Test size:", len(raw_ds["test"]))

print("\nColumns:", raw_ds["train"].column_names)

print("\nTrain sample:")
print(raw_ds["train"][0])

print("\nValidation sample:")
print(raw_ds["validation"][0])

print("\nTest sample:")
print(raw_ds["test"][0])

**Run Basic Dataset Quality Check**


In [ ]:
# Define a function to check dataset quality, completeness, duplication, and text length statistics for each split
def basic_dataset_check(ds, split_name):
    df = pd.DataFrame(ds[split_name])

    # Print the split name and total number of rows
    print(f"\n===== {split_name.upper()} =====")
    print("Rows:", len(df))
    print("Null counts:")
    print(df[["instruction", "input", "output"]].isnull().sum())

    # Count empty strings in the instruction, input, and output columns
    print("\nEmpty string counts:")
    for col in ["instruction", "input", "output"]:
        empty_count = (df[col].astype(str).str.strip() == "").sum()
        print(f"{col}: {empty_count}")

    # Print the number of fully duplicated rows in the current dataset split
    print("\nDuplicate full rows:", df.duplicated().sum())

    # Calculate input and output text lengths in words for basic length analysis
    df["input_words"] = df["input"].astype(str).apply(lambda x: len(x.split()))
    df["output_words"] = df["output"].astype(str).apply(lambda x: len(x.split()))

    # Display summary statistics for input text length
    print("\nInput word stats:")
    print(df["input_words"].describe())

    # Display summary statistics for output text length
    print("\nOutput word stats:")
    print(df["output_words"].describe())

# Run the dataset quality check for the training split
basic_dataset_check(raw_ds, "train")

# Run the dataset quality check for the validation split
basic_dataset_check(raw_ds, "validation")

# Run the dataset quality check for the test split
basic_dataset_check(raw_ds, "test")

**Load and Configure Tokenizer**


In [ ]:
# Load the tokenizer for the selected Gemma model using the Hugging Face access token
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

# Set the pad token to the EOS token if the tokenizer has no predefined pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Use right-side padding so sequences align correctly during batching
tokenizer.padding_side = "right"

# Print the final pad token configuration for verification
print("Pad token:", tokenizer.pad_token)
print("Pad token id:", tokenizer.pad_token_id)

**Build Gemma Chat Prompt Format**


In [ ]:
# Define the Gemma chat template tokens used to format instruction-input-output examples for training
START_USER = "<start_of_turn>user\n"
START_MODEL = "<start_of_turn>model\n"
END_TURN = "<end_of_turn>"

# Build the user-side prompt by combining the instruction and input triples in Gemma chat format
def build_prompt(example):
    instruction = str(example["instruction"]).strip()
    input_text = str(example["input"]).strip()

    prompt = (
        f"{START_USER}"
        f"{instruction}\n\n"
        f"{input_text}"
        f"{END_TURN}\n"
        f"{START_MODEL}"
    )
    return prompt

# Build the model-side target response by appending the end-of-turn token to the output text
def build_response(example):
    return str(example["output"]).strip() + END_TURN

# Convert each raw dataset example into prompt, response, and full training text fields
def format_example(example):
    prompt = build_prompt(example)
    response = build_response(example)
    text = prompt + response
    return {
        "prompt": prompt,
        "response": response,
        "text": text
    }

# Apply the formatting function to all dataset splits to create Gemma-ready training examples
formatted_ds = raw_ds.map(format_example)

**Inspect Formatted Training Example**


In [ ]:
print("Formatted train example:\n")
print(formatted_ds["train"][0]["text"][:3000])

print("\nPrompt only:\n")
print(formatted_ds["train"][0]["prompt"][:1500])

print("\nResponse only:\n")
print(formatted_ds["train"][0]["response"][:1500])

**Measure Prompt, Response, and Total Token Lengths**


In [ ]:
# Tokenize each prompt, response, and full text to measure sequence lengths before training
def get_token_lengths(example):
    prompt_ids = tokenizer(example["prompt"], add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(example["response"], add_special_tokens=False)["input_ids"]
    text_ids = tokenizer(example["text"], add_special_tokens=False)["input_ids"]

    return {
        "prompt_tokens": len(prompt_ids),
        "response_tokens": len(response_ids),
        "total_tokens": len(text_ids)
    }

# Apply token length calculation to all formatted dataset splits
length_ds = formatted_ds.map(get_token_lengths)

# Define a function to display descriptive statistics for token lengths in a dataset split
def show_length_stats(split_name):
    df = pd.DataFrame({
        "prompt_tokens": length_ds[split_name]["prompt_tokens"],
        "response_tokens": length_ds[split_name]["response_tokens"],
        "total_tokens": length_ds[split_name]["total_tokens"]
    })

    # Print detailed token length statistics for the selected split
    print(f"\n===== {split_name.upper()} TOKEN STATS =====")
    print(df.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))
    return df

# Show token length statistics for the training split
train_len_df = show_length_stats("train")

# Show token length statistics for the validation split
val_len_df = show_length_stats("validation")

# Show token length statistics for the test split
test_len_df = show_length_stats("test")

**Set Maximum Sequence Length from Token Statistics**

In [ ]:
# Set the maximum sequence length using the 99th percentile of training token lengths and align it to a multiple of 64
p99_total = int(np.percentile(train_len_df["total_tokens"], 99))
MAX_SEQ_LENGTH = min(2048, int(math.ceil(p99_total / 64) * 64))
MAX_SEQ_LENGTH = max(512, MAX_SEQ_LENGTH)

# Print the final sequence length selected for LoRA fine-tuning
print("Chosen MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)

**Check Potential Truncation Before Tokenization**


In [ ]:
# Check how many examples exceed the selected maximum sequence length and how many response tokens would be lost
def truncation_check(example):
    prompt_ids = tokenizer(example["prompt"], add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(example["response"], add_special_tokens=False)["input_ids"]

    full_len = len(prompt_ids) + len(response_ids)
    is_truncated = full_len > MAX_SEQ_LENGTH

    kept_response_tokens = max(0, MAX_SEQ_LENGTH - len(prompt_ids))
    lost_response_tokens = max(0, len(response_ids) - kept_response_tokens)

    return {
        "full_len": full_len,
        "is_truncated": int(is_truncated),
        "lost_response_tokens": lost_response_tokens
    }

# Apply truncation analysis to all formatted dataset splits
trunc_ds = formatted_ds.map(truncation_check)

# Print truncation statistics for each dataset split
for split in ["train", "validation", "test"]:
    df = pd.DataFrame(trunc_ds[split])
    print(f"\n===== {split.upper()} TRUNCATION CHECK =====")
    print("Truncated rows:", df["is_truncated"].sum(), "/", len(df))
    print("Average lost response tokens:", round(df["lost_response_tokens"].mean(), 2))
    print("Max lost response tokens:", df["lost_response_tokens"].max())

**Tokenize Dataset and Mask Prompt Tokens**


In [ ]:
# Tokenize each example, mask the prompt tokens in labels, and keep only the response tokens for supervised learning
def tokenize_and_mask(example):
    prompt_ids = tokenizer(
        example["prompt"],
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    response_ids = tokenizer(
        example["response"],
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    # Combine prompt and response tokens into a single input sequence
    input_ids = prompt_ids + response_ids
    attention_mask = [1] * len(input_ids)

    # Mask prompt tokens with -100 so loss is calculated only on response tokens
    labels = [-100] * len(prompt_ids) + response_ids.copy()

    # Truncate input, attention mask, and labels to the selected maximum sequence length
    input_ids = input_ids[:MAX_SEQ_LENGTH]
    attention_mask = attention_mask[:MAX_SEQ_LENGTH]
    labels = labels[:MAX_SEQ_LENGTH]

    # Count how many tokens remain supervised after masking and truncation
    supervised_tokens = sum(1 for x in labels if x != -100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "supervised_tokens": supervised_tokens
    }

# Apply tokenization and label masking to all formatted dataset splits and remove original text columns
tokenized_ds = formatted_ds.map(
    tokenize_and_mask,
    remove_columns=formatted_ds["train"].column_names
)

**Inspect Tokenized Dataset Sample**


In [ ]:
# Preview the tokenized dataset structure and inspect one training example after prompt masking
print(tokenized_ds)

# Select the first tokenized training example for inspection
sample_tok = tokenized_ds["train"][0]

# Print sequence lengths and the number of supervised target tokens
print("\nInput ids length:", len(sample_tok["input_ids"]))
print("Labels length:", len(sample_tok["labels"]))
print("Supervised tokens:", sample_tok["supervised_tokens"])

# Display the first part of the label sequence to verify prompt masking with -100 values
print("\nFirst 50 labels:")
print(sample_tok["labels"][:50])

# Display the last part of the label sequence to verify response tokens are kept for loss computation
print("\nLast 50 labels:")
print(sample_tok["labels"][-50:])

**Filter Records with Zero Supervised Tokens**


In [ ]:
# Remove examples with zero supervised response tokens to keep only trainable samples in each split
for split in ["train", "validation", "test"]:
    before = len(tokenized_ds[split])
    tokenized_ds[split] = tokenized_ds[split].filter(lambda x: x["supervised_tokens"] > 0)
    after = len(tokenized_ds[split])
    print(f"{split}: kept {after}/{before}")

**Load Base Gemma Model for Fine-Tuning**


In [ ]:
# Load the Gemma causal language model with the best supported precision and enable memory-saving training settings
from transformers import AutoModelForCausalLM

# Detect whether bfloat16 is supported on the current GPU and choose the compute dtype accordingly
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_supported else torch.float16

# Load the pretrained Gemma model for LoRA fine-tuning with automatic device placement
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=compute_dtype,
    device_map="auto"
)

# Disable cache and enable gradient checkpointing to reduce memory usage during training
model.config.use_cache = False
model.gradient_checkpointing_enable()

# Confirm that the model was loaded successfully
print("Model loaded successfully.")


**Configure and Attach LoRA Adapters**



In [ ]:
# Configure LoRA adapters for the Gemma model and wrap the base model for parameter-efficient fine-tuning
from peft import LoraConfig, get_peft_model, TaskType

# Define the LoRA setup including rank, scaling, dropout, and target projection layers
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=COMMON_LORA_R,
    lora_alpha=COMMON_LORA_ALPHA,
    lora_dropout=COMMON_LORA_DROPOUT,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

# Apply the LoRA configuration to the loaded Gemma model
model = get_peft_model(model, lora_config)

# Define a helper function to calculate trainable and total model parameters after LoRA wrapping
def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    # Print the number and percentage of trainable parameters used in LoRA fine-tuning
    print(f"Trainable params: {trainable_params:,}")
    print(f"All params: {all_params:,}")
    print(f"Trainable %: {100 * trainable_params / all_params:.4f}")

# Display the final trainable parameter summary for the LoRA model
print_trainable_parameters(model)

**Create Data Collator for Training**


In [ ]:
# Create a data collator to dynamically pad tokenized batches for efficient LoRA training
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    return_tensors="pt"
)

**Define Training Evaluation Metrics**


In [ ]:
# Compute evaluation perplexity by calculating cross-entropy loss only on supervised target tokens
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Convert model outputs and labels to PyTorch tensors
    logits = torch.tensor(logits)
    labels = torch.tensor(labels)

    # Reshape logits and labels to token-level format for loss computation
    vocab_size = logits.shape[-1]
    logits = logits.view(-1, vocab_size)
    labels = labels.view(-1)

    # Keep only valid label positions and ignore masked prompt tokens
    valid_mask = labels != -100
    logits = logits[valid_mask]
    labels = labels[valid_mask]

    # Return NaN perplexity if no supervised tokens are available
    if labels.numel() == 0:
        return {"perplexity": float("nan")}

    # Compute cross-entropy loss and convert it to perplexity
    loss_fct = torch.nn.CrossEntropyLoss()
    loss = loss_fct(logits, labels)
    ppl = torch.exp(loss).item()

    # Return perplexity as the evaluation metric
    return {"perplexity": ppl}

**Set Training Arguments**


In [ ]:
# Calculate total training steps and warmup steps, then define the full Hugging Face training configuration for LoRA fine-tuning
from transformers import TrainingArguments

# Estimate the total number of optimizer steps across all training epochs
total_train_steps = int(len(tokenized_ds["train"]) / COMMON_TRAIN_BATCH_SIZE / COMMON_GRAD_ACCUM * COMMON_EPOCHS)

# Compute the number of warmup steps based on the configured warmup ratio
warmup_steps = int(total_train_steps * COMMON_WARMUP_RATIO)

# Define the complete training arguments for supervised LoRA fine-tuning and evaluation
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=COMMON_EPOCHS,
    learning_rate=COMMON_LR,
    weight_decay=COMMON_WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    per_device_train_batch_size=COMMON_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=COMMON_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=COMMON_GRAD_ACCUM,
    logging_strategy="steps",
    logging_steps=COMMON_LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=COMMON_EVAL_STEPS,
    save_strategy="steps",
    save_steps=COMMON_SAVE_STEPS,
    save_total_limit=COMMON_SAVE_TOTAL_LIMIT,
    bf16=bf16_supported,
    fp16=not bf16_supported,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    report_to="none",
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    prediction_loss_only=True
)

**Initialize Trainer with Early Stopping**


In [ ]:
# Initialize the Hugging Face Trainer with the LoRA model, datasets, data collator, evaluation metric, and early stopping callback
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

**Save Training Run Configuration**


In [ ]:
# Save the full model, data, sequence length, training, and LoRA configuration for experiment reproducibility
run_config = {
    "model_id": MODEL_ID,
    "dataset_repo": DATASET_REPO,
    "data_files": DATA_FILES,
    "max_seq_length": MAX_SEQ_LENGTH,
    "seed": SEED,
    "epochs": COMMON_EPOCHS,
    "learning_rate": COMMON_LR,
    "weight_decay": COMMON_WEIGHT_DECAY,
    "warmup_ratio": COMMON_WARMUP_RATIO,
    "train_batch_size": COMMON_TRAIN_BATCH_SIZE,
    "eval_batch_size": COMMON_EVAL_BATCH_SIZE,
    "grad_accum": COMMON_GRAD_ACCUM,
    "lora_r": COMMON_LORA_R,
    "lora_alpha": COMMON_LORA_ALPHA,
    "lora_dropout": COMMON_LORA_DROPOUT
}

# Write the run configuration to a JSON file in the output directory
with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

# Confirm that the run configuration file has been saved
print("Saved run config.")

**Start LoRA Fine-Tuning**


In [ ]:
train_result = trainer.train()
print(train_result)

**Plot Training and Validation Loss Curves**


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Get trainer logs
logs_df = pd.DataFrame(trainer.state.log_history)

# Separate training and validation logs
train_logs = logs_df[logs_df["loss"].notna()].copy() if "loss" in logs_df.columns else pd.DataFrame()
eval_logs = logs_df[logs_df["eval_loss"].notna()].copy() if "eval_loss" in logs_df.columns else pd.DataFrame()

# Keep only needed columns
train_plot_df = train_logs[[c for c in ["step", "loss"] if c in train_logs.columns]].copy()
eval_plot_df = eval_logs[[c for c in ["step", "eval_loss"] if c in eval_logs.columns]].copy()

# Plot only training loss and validation loss
plt.figure(figsize=(10, 6))

if not train_plot_df.empty:
    plt.plot(train_plot_df["step"], train_plot_df["loss"], marker="o", label="Training Loss")

if not eval_plot_df.empty:
    plt.plot(eval_plot_df["step"], eval_plot_df["eval_loss"], marker="s", label="Validation Loss")

plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Curves")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

logs_df = pd.DataFrame(trainer.state.log_history)

train_logs = logs_df[logs_df["loss"].notna()].copy() if "loss" in logs_df.columns else pd.DataFrame()
eval_logs = logs_df[logs_df["eval_loss"].notna()].copy() if "eval_loss" in logs_df.columns else pd.DataFrame()

plt.figure(figsize=(10, 6))

if not train_logs.empty and "epoch" in train_logs.columns:
    plt.plot(train_logs["epoch"], train_logs["loss"], marker="o", label="Training Loss")

if not eval_logs.empty and "epoch" in eval_logs.columns:
    plt.plot(eval_logs["epoch"], eval_logs["eval_loss"], marker="s", label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Curves")
plt.legend()
plt.grid(True)
plt.show()

**Save LoRA Adapter and Tokenizer**


In [ ]:
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved adapter to:", ADAPTER_DIR)

**Save Run Metadata and Configuration JSON**


In [ ]:
import json

run_config = {
    "method": "LoRA",
    "model_id": MODEL_ID,
    "dataset_repo": DATASET_REPO,
    "max_seq_length": MAX_SEQ_LENGTH,
    "epochs": COMMON_EPOCHS,
    "learning_rate": COMMON_LR,
    "train_batch_size": COMMON_TRAIN_BATCH_SIZE,
    "eval_batch_size": COMMON_EVAL_BATCH_SIZE,
    "grad_accum": COMMON_GRAD_ACCUM,
    "lora_r": COMMON_LORA_R
}

with open(f"{OUTPUT_DIR}/run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

print("Run config saved")

**Preview Test Example for Inference**


In [ ]:
test_example = raw_ds["test"][0]

print("INSTRUCTION:\n")
print(test_example["instruction"])

print("\nINPUT TRIPLES:\n")
print(test_example["input"])

print("\nREFERENCE OUTPUT:\n")
print(test_example["output"])

**Define KG-to-Text Generation Function**


In [ ]:
# Define the text generation function used to test the fine-tuned Gemma LoRA model on KG-to-text inputs
import torch

# Reuse the Gemma chat template tokens for inference prompt formatting
START_USER = "<start_of_turn>user\n"
START_MODEL = "<start_of_turn>model\n"
END_TURN = "<end_of_turn>"

# Build a prompt from instruction and triples, generate output text, and clean the decoded response
def generate_kg_text(
    instruction,
    triples_text,
    model,
    tokenizer,
    max_new_tokens=400,
    min_new_tokens=80
):
    # Construct the full Gemma-style inference prompt
    prompt = (
        f"{START_USER}"
        f"{instruction.strip()}\n\n"
        f"{triples_text.strip()}"
        f"{END_TURN}\n"
        f"{START_MODEL}"
    )

    # Tokenize the prompt and move it to the same device as the model
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Get the end-of-turn token ID to stop generation at the correct boundary
    end_turn_ids = tokenizer.encode(END_TURN, add_special_tokens=False)
    end_turn_id = end_turn_ids[0] if len(end_turn_ids) > 0 else tokenizer.eos_token_id

    # Generate the model response without sampling for deterministic output
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            do_sample=False,
            temperature=1.0,
            eos_token_id=end_turn_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.05
        )

    # Decode the generated tokens back into text
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Remove the prompt prefix and keep only the generated model response
    if START_MODEL in decoded:
        decoded = decoded.split(START_MODEL, 1)[1]

    # Stop the output at the first end-of-turn token
    if END_TURN in decoded:
        decoded = decoded.split(END_TURN, 1)[0]

    # Return the cleaned generated text
    return decoded.strip()

**Run Single Test Prediction**


In [ ]:
# Generate a prediction on one test example and compare the model output with the reference text
test_example = raw_ds["test"][0]

# Run inference on the selected test example using the fine-tuned LoRA model
prediction = generate_kg_text(
    test_example["instruction"],
    test_example["input"],
    model,
    tokenizer,
    max_new_tokens=400,
    min_new_tokens=100
)

# Print the ground-truth reference output from the test dataset
print("REFERENCE:\n")
print(test_example["output"])

# Print the generated prediction from the fine-tuned model
print("\nPREDICTION:\n")
print(prediction)

**Generate predictions for full test set**


In [ ]:
# Install evaluation libraries and import all required packages for full test set generation and metric computation
!pip -q install evaluate rouge_score bert-score

import os
import gc
import json
import torch
import pandas as pd
from tqdm.auto import tqdm
import evaluate

# Load BLEU, ROUGE, and BERTScore metrics for generation quality evaluation
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

# Define the Gemma chat template tokens for formatting inference prompts
START_USER = "<start_of_turn>user\n"
START_MODEL = "<start_of_turn>model\n"
END_TURN = "<end_of_turn>"

# Define the generation function used to produce predictions from the fine-tuned model
def generate_kg_text(
    instruction,
    triples_text,
    model,
    tokenizer,
    max_new_tokens=500,
    min_new_tokens=100
):
    # Construct the inference prompt using instruction and KG triples
    prompt = (
        f"{START_USER}"
        f"{instruction.strip()}\n\n"
        f"{triples_text.strip()}"
        f"{END_TURN}\n"
        f"{START_MODEL}"
    )

    # Tokenize the prompt and move inputs to the model device
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Get the end-of-turn token ID to control generation stopping
    end_turn_ids = tokenizer.encode(END_TURN, add_special_tokens=False)
    end_turn_id = end_turn_ids[0] if len(end_turn_ids) > 0 else tokenizer.eos_token_id

    # Generate the model output deterministically without sampling
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            do_sample=False,
            temperature=1.0,
            eos_token_id=end_turn_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.05
        )

    # Decode the generated tokens into text
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Remove the prompt prefix and keep only the generated model response
    if START_MODEL in decoded:
        decoded = decoded.split(START_MODEL, 1)[1]

    # Stop the decoded text at the first end-of-turn token
    if END_TURN in decoded:
        decoded = decoded.split(END_TURN, 1)[0]

    return decoded.strip()


# Create an empty list to store generated predictions and references for the full test set
results = []

# Generate predictions for every example in the test split
for i in tqdm(range(len(raw_ds["test"])), desc="Generating test predictions"):
    example = raw_ds["test"][i]

    # Run model inference on the current test example
    prediction = generate_kg_text(
        example["instruction"],
        example["input"],
        model,
        tokenizer,
        max_new_tokens=500,
        min_new_tokens=100
    )

    # Store the test example metadata, reference, and generated prediction
    results.append({
        "index": i,
        "instruction": example["instruction"],
        "input": example["input"],
        "reference": example["output"],
        "prediction": prediction
    })

    # Periodically clean memory during long test generation runs
    if i % 10 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Convert the generated test predictions into a DataFrame
results_df = pd.DataFrame(results)

# Save the full test set predictions to a CSV file
predictions_path = os.path.join(OUTPUT_DIR, "test_predictions_full.csv")
results_df.to_csv(predictions_path, index=False)
print("Saved predictions to:", predictions_path)

# Display a preview of the generated test predictions
display(results_df.head(3))


**Prepare Predictions and References for Metrics and calculate metrics**


In [ ]:
# Prepare prediction and reference text lists for corpus-level evaluation metrics
predictions = results_df["prediction"].fillna("").astype(str).tolist()
references = results_df["reference"].fillna("").astype(str).tolist()

# Convert references into BLEU-compatible nested list format
references_for_bleu = [[ref] for ref in references]

# Compute BLEU score and related corpus-level precision statistics
bleu_result = bleu_metric.compute(
    predictions=predictions,
    references=references_for_bleu
)

# Compute ROUGE scores using stemmed matching
rouge_result = rouge_metric.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

# Compute BERTScore precision, recall, and F1 for semantic similarity evaluation
bertscore_result = bertscore_metric.compute(
    predictions=predictions,
    references=references,
    lang="en"
)

# Calculate the average BERTScore precision, recall, and F1 across all test examples
avg_bertscore_precision = sum(bertscore_result["precision"]) / len(bertscore_result["precision"])
avg_bertscore_recall = sum(bertscore_result["recall"]) / len(bertscore_result["recall"])
avg_bertscore_f1 = sum(bertscore_result["f1"]) / len(bertscore_result["f1"])

# Calculate average token lengths of predictions and references for length analysis
pred_token_lens = [len(tokenizer.encode(p, add_special_tokens=False)) for p in predictions]
ref_token_lens = [len(tokenizer.encode(r, add_special_tokens=False)) for r in references]

avg_pred_len = sum(pred_token_lens) / len(pred_token_lens)
avg_ref_len = sum(ref_token_lens) / len(ref_token_lens)
length_ratio = avg_pred_len / avg_ref_len if avg_ref_len > 0 else 0.0

# Build the final evaluation summary combining BLEU, ROUGE, BERTScore, and length statistics
metrics_summary = {
    "test_size": len(results_df),

    "bleu": bleu_result["bleu"],
    "bleu_precisions": bleu_result["precisions"],
    "bleu_brevity_penalty": bleu_result["brevity_penalty"],
    "bleu_length_ratio": bleu_result["length_ratio"],
    "bleu_translation_length": bleu_result["translation_length"],
    "bleu_reference_length": bleu_result["reference_length"],

    "rouge1": rouge_result["rouge1"],
    "rouge2": rouge_result["rouge2"],
    "rougeL": rouge_result["rougeL"],
    "rougeLsum": rouge_result["rougeLsum"],

    "bertscore_precision": avg_bertscore_precision,
    "bertscore_recall": avg_bertscore_recall,
    "bertscore_f1": avg_bertscore_f1,

    "avg_prediction_token_length": avg_pred_len,
    "avg_reference_token_length": avg_ref_len,
    "prediction_reference_length_ratio": length_ratio
}

# Convert the final metrics summary into a DataFrame for display
metrics_df = pd.DataFrame([metrics_summary])
display(metrics_df)

# Save the full evaluation summary to a JSON file
metrics_path = os.path.join(OUTPUT_DIR, "test_metrics_summary.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)

# Confirm that the evaluation metrics file has been saved
print("Saved metrics to:", metrics_path)

# Add per-example token length statistics to the predictions DataFrame
results_df["prediction_token_length"] = pred_token_lens
results_df["reference_token_length"] = ref_token_lens
results_df["length_ratio"] = results_df["prediction_token_length"] / results_df["reference_token_length"].replace(0, 1)

# Save the updated predictions file with added length statistics
results_df.to_csv(predictions_path, index=False)
print("Updated predictions file with length stats:", predictions_path)

**Parse Triples and Build Factual Evaluation Logic**


In [ ]:
# Import libraries required to compute fact-based faithfulness and hallucination metrics from generated outputs
import re
import json
import pandas as pd
import numpy as np

# Extract structured facts from KG triple text into a predicate-wise fact dictionary
def extract_facts_from_triples(triples_text):
    facts = {
        "hasName": [],
        "hasDrugBankID": [],
        "hasState": [],
        "hasUNII": [],
        "hasCASNumber": [],
        "hasAverageMass": [],
        "hasMonoisotopicMass": [],
        "hasGroup": [],
        "hasSynonym": []
    }

    # Parse each triple line and store supported facts by predicate
    for line in str(triples_text).splitlines():
        line = line.strip()
        m = re.match(r"\((.*?),\s*([^,]+),\s*(.*?)\)$", line)
        if m:
            _, predicate, obj = m.groups()
            predicate = predicate.strip()
            obj = obj.strip()

            if predicate in facts:
                facts[predicate].append(obj)

    return facts

# Normalize text values for case-insensitive and whitespace-stable fact matching
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

# Normalize numeric strings by lowercasing and removing commas for number matching
def normalize_number_string(text):
    text = normalize_text(text)
    text = text.replace(",", "")
    return text

# Check whether a gold fact from the triples is supported in the generated prediction text
def fact_supported_in_prediction(predicate, obj, prediction_text):
    pred = normalize_text(prediction_text)
    obj_norm = normalize_text(obj)

    # Check exact text match for categorical and identifier predicates
    if predicate in ["hasName", "hasDrugBankID", "hasState", "hasUNII", "hasCASNumber", "hasGroup", "hasSynonym"]:
        return obj_norm in pred

    # Check numeric fact presence for molecular mass predicates
    if predicate in ["hasAverageMass", "hasMonoisotopicMass"]:
        obj_num = normalize_number_string(obj)
        pred_num = normalize_number_string(prediction_text)
        return obj_num in pred_num

    return False

# Extract fact-like candidates from the generated prediction that can be checked against the triples
def extract_prediction_fact_candidates(prediction_text):
    pred = str(prediction_text)

    candidates = []

    # Extract DrugBank IDs mentioned in the prediction
    for m in re.findall(r"\bDB\d{5}\b", pred):
        candidates.append(("hasDrugBankID", m))

    # Extract UNII-like identifiers mentioned in the prediction
    for m in re.findall(r"\b[A-Z0-9]{10}\b", pred):
        candidates.append(("hasUNII", m))

    # Extract CAS numbers mentioned in the prediction
    for m in re.findall(r"\b\d{2,7}-\d{2}-\d\b", pred):
        candidates.append(("hasCASNumber", m))

    # Extract simple physical state mentions from the prediction
    for state in ["solid", "liquid", "gas"]:
        if re.search(rf"\b{state}\b", pred.lower()):
            candidates.append(("hasState", state))

    # Extract numeric values that may correspond to molecular masses
    for m in re.findall(r"\b\d+\.\d+\b", pred):
        candidates.append(("numeric_value", m))

    return candidates

# Compute fact recall, fact precision, fact F1, and hallucination rate for a single prediction
def compute_fact_metrics_for_example(triples_text, prediction_text):
    facts = extract_facts_from_triples(triples_text)

    # Check how many gold facts from the triples are recovered in the generated prediction
    gold_checks = []
    for predicate in ["hasName", "hasDrugBankID", "hasState", "hasUNII", "hasCASNumber", "hasAverageMass", "hasMonoisotopicMass"]:
        for obj in facts.get(predicate, []):
            supported = fact_supported_in_prediction(predicate, obj, prediction_text)
            gold_checks.append((predicate, obj, supported))

    gold_total = len(gold_checks)
    gold_supported = sum(1 for _, _, ok in gold_checks if ok)
    fact_recall = gold_supported / gold_total if gold_total > 0 else np.nan

    # Extract fact candidates from the prediction and verify whether they are supported by the input triples
    pred_candidates = extract_prediction_fact_candidates(prediction_text)
    pred_checks = []

    fact_set = set()
    for predicate, values in facts.items():
        for v in values:
            fact_set.add((predicate, normalize_text(v)))
            if predicate in ["hasAverageMass", "hasMonoisotopicMass"]:
                fact_set.add(("numeric_value", normalize_number_string(v)))

    # Mark each predicted fact candidate as supported or unsupported by the input triples
    for predicate, obj in pred_candidates:
        key = (predicate, normalize_text(obj)) if predicate != "numeric_value" else ("numeric_value", normalize_number_string(obj))
        pred_checks.append((predicate, obj, key in fact_set))

    pred_total = len(pred_checks)
    pred_supported = sum(1 for _, _, ok in pred_checks if ok)

    fact_precision = pred_supported / pred_total if pred_total > 0 else np.nan
    hallucination_rate = 1 - fact_precision if pred_total > 0 else np.nan

    # Compute fact-level F1 score when both precision and recall are available
    if pd.notna(fact_precision) and pd.notna(fact_recall) and (fact_precision + fact_recall) > 0:
        fact_f1 = 2 * fact_precision * fact_recall / (fact_precision + fact_recall)
    else:
        fact_f1 = np.nan

    return {
        "gold_fact_total": gold_total,
        "gold_fact_supported": gold_supported,
        "fact_recall": fact_recall,

        "pred_fact_total": pred_total,
        "pred_fact_supported": pred_supported,
        "fact_precision": fact_precision,
        "fact_f1": fact_f1,
        "hallucination_rate": hallucination_rate
    }

# Compute fact-based faithfulness metrics for every prediction in the full test results
hallucination_rows = []

for i, row in results_df.iterrows():
    metrics = compute_fact_metrics_for_example(row["input"], row["prediction"])
    hallucination_rows.append(metrics)

# Merge the hallucination metrics with the original test prediction results
hall_df = pd.DataFrame(hallucination_rows)
results_with_hall = pd.concat([results_df.reset_index(drop=True), hall_df.reset_index(drop=True)], axis=1)

# Save the detailed per-example hallucination analysis file
hall_path = os.path.join(OUTPUT_DIR, "test_predictions_with_hallucination_metrics.csv")
results_with_hall.to_csv(hall_path, index=False)
print("Saved detailed hallucination metrics to:", hall_path)

# Build summary-level hallucination and fact-faithfulness metrics across the full test set
hall_summary = {
    "avg_fact_precision": float(hall_df["fact_precision"].dropna().mean()),
    "avg_fact_recall": float(hall_df["fact_recall"].dropna().mean()),
    "avg_fact_f1": float(hall_df["fact_f1"].dropna().mean()),
    "avg_hallucination_rate": float(hall_df["hallucination_rate"].dropna().mean()),

    "examples_with_precision_score": int(hall_df["fact_precision"].notna().sum()),
    "examples_with_recall_score": int(hall_df["fact_recall"].notna().sum())
}

# Print the final hallucination-related summary metrics
print("\n===== HALLUCINATION-RELATED METRICS =====")
for k, v in hall_summary.items():
    print(f"{k}: {v}")

# Save the hallucination summary metrics to a JSON file
hall_summary_path = os.path.join(OUTPUT_DIR, "hallucination_metrics_summary.json")
with open(hall_summary_path, "w") as f:
    json.dump(hall_summary, f, indent=2)

print("Saved hallucination summary to:", hall_summary_path)

**Compute Exact Match Score**


In [ ]:
results_df["exact_match"] = (
    results_df["prediction"].astype(str).str.strip()
    == results_df["reference"].astype(str).str.strip()
)

exact_match_rate = results_df["exact_match"].mean()
print("Exact match rate:", exact_match_rate)
print("Exact match count:", results_df["exact_match"].sum(), "/", len(results_df))

**Check Train–Test Input Overlap**


In [ ]:
train_inputs = set(pd.Series(raw_ds["train"]["input"]).astype(str).str.strip())
test_inputs = pd.Series(raw_ds["test"]["input"]).astype(str).str.strip()

overlap_count = test_inputs.isin(train_inputs).sum()
print("Test inputs overlapping with train:", overlap_count, "/", len(test_inputs))

**Check Train–Test Output Overlap**


In [ ]:
train_outputs = set(pd.Series(raw_ds["train"]["output"]).astype(str).str.strip())
test_outputs = pd.Series(raw_ds["test"]["output"]).astype(str).str.strip()

overlap_out_count = test_outputs.isin(train_outputs).sum()
print("Test outputs overlapping with train:", overlap_out_count, "/", len(test_outputs))

**Save Final Evaluation Results and Reports**


In [ ]:
import os
import json
import pandas as pd

#CREATE OUTPUT PATHS
FINAL_RESULTS_DIR = os.path.join(OUTPUT_DIR, "final_results")
os.makedirs(FINAL_RESULTS_DIR, exist_ok=True)

final_json_path = os.path.join(FINAL_RESULTS_DIR, "lora_final_results.json")
final_csv_path = os.path.join(FINAL_RESULTS_DIR, "lora_final_results_table.csv")

#SAFE GET HELPERS
def safe_get(d, key, default=None):
    if isinstance(d, dict):
        return d.get(key, default)
    return default


#COLLECT TEXT GENERATION METRICS
text_metrics = {}

if "metrics_summary" in globals():
    text_metrics = {
        "test_size": safe_get(metrics_summary, "test_size"),
        "bleu": safe_get(metrics_summary, "bleu"),
        "bleu_brevity_penalty": safe_get(metrics_summary, "bleu_brevity_penalty"),
        "bleu_length_ratio": safe_get(metrics_summary, "bleu_length_ratio"),
        "rouge1": safe_get(metrics_summary, "rouge1"),
        "rouge2": safe_get(metrics_summary, "rouge2"),
        "rougeL": safe_get(metrics_summary, "rougeL"),
        "rougeLsum": safe_get(metrics_summary, "rougeLsum"),
        "bertscore_precision": safe_get(metrics_summary, "bertscore_precision"),
        "bertscore_recall": safe_get(metrics_summary, "bertscore_recall"),
        "bertscore_f1": safe_get(metrics_summary, "bertscore_f1"),
        "avg_prediction_token_length": safe_get(metrics_summary, "avg_prediction_token_length"),
        "avg_reference_token_length": safe_get(metrics_summary, "avg_reference_token_length"),
        "prediction_reference_length_ratio": safe_get(metrics_summary, "prediction_reference_length_ratio"),
    }

#COLLECT HALLUCINATION METRICS
hall_metrics = {}

if "hall_summary" in globals():
    hall_metrics = {
        "avg_fact_precision": safe_get(hall_summary, "avg_fact_precision"),
        "avg_fact_recall": safe_get(hall_summary, "avg_fact_recall"),
        "avg_fact_f1": safe_get(hall_summary, "avg_fact_f1"),
        "avg_hallucination_rate": safe_get(hall_summary, "avg_hallucination_rate"),
        "examples_with_precision_score": safe_get(hall_summary, "examples_with_precision_score"),
        "examples_with_recall_score": safe_get(hall_summary, "examples_with_recall_score"),
    }

#EXACT MATCH STATS

exact_match_stats = {}

if "results_df" in globals():
    if "exact_match" not in results_df.columns:
        results_df["exact_match"] = (
            results_df["prediction"].astype(str).str.strip()
            == results_df["reference"].astype(str).str.strip()
        )

    exact_match_rate = results_df["exact_match"].mean()
    exact_match_count = int(results_df["exact_match"].sum())

    exact_match_stats = {
        "exact_match_rate": float(exact_match_rate),
        "exact_match_count": exact_match_count
    }

#OVERLAP / LEAKAGE CHECKS
#Recompute safely from raw_ds
overlap_stats = {}

if "raw_ds" in globals():
    train_inputs = set(pd.Series(raw_ds["train"]["input"]).astype(str).str.strip())
    test_inputs = pd.Series(raw_ds["test"]["input"]).astype(str).str.strip()
    overlap_input_count = int(test_inputs.isin(train_inputs).sum())

    train_outputs = set(pd.Series(raw_ds["train"]["output"]).astype(str).str.strip())
    test_outputs = pd.Series(raw_ds["test"]["output"]).astype(str).str.strip()
    overlap_output_count = int(test_outputs.isin(train_outputs).sum())

    overlap_stats = {
        "test_input_overlap_with_train": overlap_input_count,
        "test_output_overlap_with_train": overlap_output_count,
        "test_input_overlap_rate": float(overlap_input_count / len(test_inputs)),
        "test_output_overlap_rate": float(overlap_output_count / len(test_outputs)),
    }

# TRAINING SUMMARY
training_stats = {}

if "trainer" in globals():
    training_stats = {
        "global_step": trainer.state.global_step,
        "best_metric": trainer.state.best_metric,
        "best_checkpoint": trainer.state.best_model_checkpoint,
    }

if "train_result" in globals() and hasattr(train_result, "metrics"):
    training_stats.update({
        "train_runtime": train_result.metrics.get("train_runtime"),
        "train_samples_per_second": train_result.metrics.get("train_samples_per_second"),
        "train_steps_per_second": train_result.metrics.get("train_steps_per_second"),
        "train_loss": train_result.metrics.get("train_loss"),
    })

#RUN CONFIG SUMMARY
config_stats = {}

for var_name in [
    "MODEL_ID",
    "DATASET_REPO",
    "MAX_SEQ_LENGTH",
    "COMMON_EPOCHS",
    "COMMON_LR",
    "COMMON_TRAIN_BATCH_SIZE",
    "COMMON_EVAL_BATCH_SIZE",
    "COMMON_GRAD_ACCUM",
    "COMMON_LORA_R",
    "COMMON_LORA_ALPHA",
    "COMMON_LORA_DROPOUT"
]:
    if var_name in globals():
        config_stats[var_name] = globals()[var_name]

#FINAL COMBINED RESULTS
final_results = {
    "method": "LoRA",
    "model_id": globals().get("MODEL_ID", None),
    "dataset_repo": globals().get("DATASET_REPO", None),

    "config": config_stats,
    "training_summary": training_stats,
    "text_generation_metrics": text_metrics,
    "hallucination_metrics": hall_metrics,
    "exact_match_stats": exact_match_stats,
    "overlap_checks": overlap_stats
}

#SAVE JSON
with open(final_json_path, "w") as f:
    json.dump(final_results, f, indent=2)

print("Saved final JSON results to:", final_json_path)

#SAVE FLAT TABLE CSV
flat_table = {
    "method": "LoRA",
    "model_id": globals().get("MODEL_ID", None),
    "dataset_repo": globals().get("DATASET_REPO", None),

    "max_seq_length": globals().get("MAX_SEQ_LENGTH", None),
    "epochs": globals().get("COMMON_EPOCHS", None),
    "learning_rate": globals().get("COMMON_LR", None),
    "train_batch_size": globals().get("COMMON_TRAIN_BATCH_SIZE", None),
    "eval_batch_size": globals().get("COMMON_EVAL_BATCH_SIZE", None),
    "grad_accum": globals().get("COMMON_GRAD_ACCUM", None),
    "lora_r": globals().get("COMMON_LORA_R", None),
    "lora_alpha": globals().get("COMMON_LORA_ALPHA", None),
    "lora_dropout": globals().get("COMMON_LORA_DROPOUT", None),

    "global_step": training_stats.get("global_step"),
    "best_metric": training_stats.get("best_metric"),
    "train_runtime": training_stats.get("train_runtime"),
    "train_loss": training_stats.get("train_loss"),

    "test_size": text_metrics.get("test_size"),
    "bleu": text_metrics.get("bleu"),
    "bleu_brevity_penalty": text_metrics.get("bleu_brevity_penalty"),
    "bleu_length_ratio": text_metrics.get("bleu_length_ratio"),
    "rouge1": text_metrics.get("rouge1"),
    "rouge2": text_metrics.get("rouge2"),
    "rougeL": text_metrics.get("rougeL"),
    "rougeLsum": text_metrics.get("rougeLsum"),
    "bertscore_precision": text_metrics.get("bertscore_precision"),
    "bertscore_recall": text_metrics.get("bertscore_recall"),
    "bertscore_f1": text_metrics.get("bertscore_f1"),
    "avg_prediction_token_length": text_metrics.get("avg_prediction_token_length"),
    "avg_reference_token_length": text_metrics.get("avg_reference_token_length"),
    "prediction_reference_length_ratio": text_metrics.get("prediction_reference_length_ratio"),

    "avg_fact_precision": hall_metrics.get("avg_fact_precision"),
    "avg_fact_recall": hall_metrics.get("avg_fact_recall"),
    "avg_fact_f1": hall_metrics.get("avg_fact_f1"),
    "avg_hallucination_rate": hall_metrics.get("avg_hallucination_rate"),

    "exact_match_rate": exact_match_stats.get("exact_match_rate"),
    "exact_match_count": exact_match_stats.get("exact_match_count"),

    "test_input_overlap_with_train": overlap_stats.get("test_input_overlap_with_train"),
    "test_output_overlap_with_train": overlap_stats.get("test_output_overlap_with_train"),
    "test_input_overlap_rate": overlap_stats.get("test_input_overlap_rate"),
    "test_output_overlap_rate": overlap_stats.get("test_output_overlap_rate"),
}

final_df = pd.DataFrame([flat_table])
final_df.to_csv(final_csv_path, index=False)

print("Saved final CSV table to:", final_csv_path)
display(final_df)


root_json_path = os.path.join(OUTPUT_DIR, "lora_final_results.json")
root_csv_path = os.path.join(OUTPUT_DIR, "lora_final_results_table.csv")

with open(root_json_path, "w") as f:
    json.dump(final_results, f, indent=2)

final_df.to_csv(root_csv_path, index=False)

print("Also saved root JSON to:", root_json_path)
print("Also saved root CSV to:", root_csv_path)

**Deploy in HuggingFace**

In [ ]:
!pip -q install -U huggingface_hub peft transformers safetensors

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

In [ ]:
# Define the base model, Hugging Face repository names, and local paths for adapter and merged model publishing
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import HfApi

MODEL_ID = "google/gemma-1.1-2b-it"

ADAPTER_REPO_ID = "BSVGK/gemma-1.1-2b-it-drugbank-kg2text-lora-v2"
MERGED_REPO_ID  = "BSVGK/gemma-1.1-2b-it-drugbank-kg2text-merged-v2"

# Define the local output directories where the trained LoRA adapter and merged model are stored
OUTPUT_DIR = "/content/drive/MyDrive/Depixen/gemma-1.1-2b-it-drugbank-kg2text/LORA_OUTPUTS"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")
MERGED_DIR = os.path.join(OUTPUT_DIR, "merged")

# Create the merged model directory if it does not already exist
os.makedirs(MERGED_DIR, exist_ok=True)

# Print the resolved output paths and list adapter files for verification before upload or merge
print("OUTPUT_DIR:", OUTPUT_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("MERGED_DIR:", MERGED_DIR)
print("Adapter files:", sorted(os.listdir(ADAPTER_DIR)))


**Create Adapter Model Card**

In [ ]:
adapter_readme = f"""---
base_model: {MODEL_ID}
library_name: peft
tags:
- text-generation
- lora
- kg-to-text
- drugbank
- gemma
---

# Gemma 1.1 2B IT DrugBank KG-to-Text LoRA Adapter v2

This repository contains the LoRA adapter fine-tuned for converting DrugBank knowledge graph triples into natural language descriptions.

## Base model
- {MODEL_ID}

## Task
- Input: DrugBank KG triples
- Output: Natural language drug summary

## Notes
This repo stores adapter weights only.
"""

with open(os.path.join(ADAPTER_DIR, "README.md"), "w") as f:
    f.write(adapter_readme)

print("Adapter README saved.")

**Push LoRA Adapter Model to Hugging Face**


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

api = HfApi()
api.create_repo(repo_id=ADAPTER_REPO_ID, repo_type="model", token=HF_TOKEN, exist_ok=True)

tokenizer.push_to_hub(ADAPTER_REPO_ID, token=HF_TOKEN)
trainer.model.push_to_hub(ADAPTER_REPO_ID, token=HF_TOKEN)

for filename in os.listdir(ADAPTER_DIR):
    full_path = os.path.join(ADAPTER_DIR, filename)
    if os.path.isfile(full_path):
        try:
            api.upload_file(
                path_or_fileobj=full_path,
                path_in_repo=filename,
                repo_id=ADAPTER_REPO_ID,
                repo_type="model",
                token=HF_TOKEN,
            )
        except Exception as e:
            print(f"Skipped {filename}: {e}")

print("LoRA adapter uploaded to:", ADAPTER_REPO_ID)

**Load Base Model and Merge LoRA Weights**


In [ ]:
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_supported else torch.float16

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=compute_dtype,
    device_map="auto"
)

peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

print("Merged model saved to:", MERGED_DIR)
print("Merged files:", sorted(os.listdir(MERGED_DIR)))

**Create Merged Model Card**


In [ ]:
merged_readme = f"""---
base_model: {MODEL_ID}
library_name: transformers
tags:
- text-generation
- kg-to-text
- drugbank
- gemma
---

# Gemma 1.1 2B IT DrugBank KG-to-Text Merged Model v2

This repository contains the merged full model created from:
- base model: {MODEL_ID}
- LoRA adapter: {ADAPTER_REPO_ID}

## Task
Convert DrugBank knowledge graph triples into natural language drug summaries.

## Notes
This repo stores a standalone merged model for direct inference and deployment.
"""

with open(os.path.join(MERGED_DIR, "README.md"), "w") as f:
    f.write(merged_readme)

print("Merged README saved.")

**Push Merged Full Model to Hugging Face**


In [ ]:
api.create_repo(repo_id=MERGED_REPO_ID, repo_type="model", token=HF_TOKEN, exist_ok=True)

merged_model.push_to_hub(MERGED_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(MERGED_REPO_ID, token=HF_TOKEN)

for filename in os.listdir(MERGED_DIR):
    full_path = os.path.join(MERGED_DIR, filename)
    if os.path.isfile(full_path):
        try:
            api.upload_file(
                path_or_fileobj=full_path,
                path_in_repo=filename,
                repo_id=MERGED_REPO_ID,
                repo_type="model",
                token=HF_TOKEN,
            )
        except Exception as e:
            print(f"Skipped {filename}: {e}")

print("Merged model uploaded to:", MERGED_REPO_ID)

**Print Final Hugging Face Repository Links**

In [ ]:
print("Adapter repo:", ADAPTER_REPO_ID)
print("Merged repo:", MERGED_REPO_ID)